# Imports

In [ ]:
import json
import os
import pathlib
import sys

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Needed because this notebook is run from a directory inside `src`
sys.path.insert(0, str(pathlib.PurePath('..')))

In [ ]:
import linear_geodesic_optimization.driver as driver
from analysis import loss_bounds

# Data Parsing

In [ ]:
# First, load and parse the config file
data_dir = pathlib.PurePath('..', '..', 'data')
output_dir = pathlib.PurePath('..', '..', 'outputs')
config_file = data_dir / 'toy' / 'routing_with_volumes' / 'config' / 'traffic_reroute_link_outage_ridges_tall_smooth_120.json'

output_index = 1

arguments, _, settings, defaults = driver.load_config(config_file)

output_format = driver.get_output_format(settings, defaults)
driver.assign_output_directories(arguments, settings, output_format, output_directory=output_dir, existence_check='must_exist')

initialization = driver.get_initialization(settings)
driver.assign_initialization_file_paths(arguments, initialization)

argument_sets = arguments[output_index][(1 if settings['initialization'] == 'first' else 0):]

In [ ]:
# Grab the per-iteration losses from the snapshots (<iteration>.json)
losses = []
for argument_set in argument_sets:
    losses_i = {}
    lambda_curvature = argument_set['lambda_curvature']
    lambda_smooth = argument_set['lambda_smooth']
    argument_directory_output = pathlib.PurePath(argument_set['directory_output'])
    for filename in sorted(os.listdir(argument_directory_output)):
        filepath = argument_directory_output / filename
        if filepath.suffix != '.json' or not filepath.stem.isdigit():
            continue
        iteration = int(filepath.stem)
        with open(filepath, 'r') as f:
            output = json.load(f)
            loss_curvature = output['L_curvature']
            loss_smooth = output['L_smooth']
            losses_i[iteration] = {
                'L_curvature': loss_curvature,
                'L_smooth': loss_smooth,
                'L_total': lambda_curvature * loss_curvature + lambda_smooth * loss_smooth,
            }
    losses.append(losses_i)

## Lower bound on the curvature loss

$L_{\text{curvature}} = \sum_i w_i (K_{v_i} - \kappa_i)^2$ over the (mesh vertex, target) pairs of the edges' tubes. A vertex in several tubes gets several targets but has one $K$, so its cost is at least $S_2 - S_1^2/S_0$ (with $S_k = \sum w_i \kappa_i^k$ over its pairs), reached at the weighted mean $K^* = S_1/S_0$. Boundary vertices have $K = 0$ and cost $S_2$. The sum $L_{\text{LB}}$ bounds $L_{\text{curvature}}$ from below, and $\lambda_{\text{curvature}} L_{\text{LB}}$ bounds the total loss.

The bound is in practice reachable: L-BFGS on $L_{\text{curvature}}$ alone converges to within about $10^{-7}$ of it. So $L_{\text{curvature}} - L_{\text{LB}}$ is the misfit a better surface could remove. See `analysis/loss_bounds.py`, which also runs as a script on batch directories.

In [ ]:
# Recompute each run's final losses with the loss computer that made
# it, and the lower bound with per-vertex optimal targets
results = [
    loss_bounds.analyze_run(argument_set['directory_output'], arrays=True)
    for argument_set in argument_sets
]
loss_bounds.print_table(results)

target_curvatures = [r['k_star'].reshape((r['sides'], r['sides'])) for r in results]
actual_curvatures = [r['kappa_G'].reshape((r['sides'], r['sides'])) for r in results]

# Plotting
## Loss vs. Iteration

In [ ]:
fig, axs = plt.subplots(len(losses), 3, figsize=(12, 3 * len(losses)), sharex='all', sharey='col')
axs = np.atleast_2d(axs)
fig.suptitle('Loss vs. Iteration (dashed: lower bound)')
for col, title in enumerate([r'$L_{\text{curvature}}$', r'$L_{\text{smooth}}$', r'$L_{\text{total}}$']):
    axs[0][col].set_title(title)
for row, (losses_i, result, axs_i) in enumerate(zip(losses, results, axs)):
    axs_i[0].set_ylabel(f'$t = {row}$')
    x = list(sorted(losses_i.keys()))
    for ax, key in zip(axs_i, ['L_curvature', 'L_smooth', 'L_total']):
        ax.plot(x, [losses_i[xj][key] for xj in x], 'b-')
    axs_i[0].axhline(result['L_LB'], color='r', linestyle='--')
    axs_i[2].axhline(result['lambda_curvature'] * result['L_LB'], color='r', linestyle='--')
for ax in axs[-1]:
    ax.set_xlabel('iteration')
fig.subplots_adjust(hspace=0.5, wspace=0.4, top=0.95)
plt.show()

## Loss vs. Timestamp

In [ ]:
t = list(range(len(results)))
fig, axs = plt.subplots(4, 1, figsize=(6, 11))
fig.suptitle(r'Final loss vs. $t$ (dashed: lower bound)')
panels = [
    (r'$L_{\text{curvature}}$', [r['L_curvature'] for r in results], [r['L_LB'] for r in results]),
    (r'$L_{\text{smooth}}$', [r['L_smooth'] for r in results], None),
    (r'$L_{\text{total}}$', [r['total'] for r in results], [r['lambda_curvature'] * r['L_LB'] for r in results]),
    (r'$L_{\text{curvature}} - L_{\text{LB}}$ (removable misfit)', [r['excess'] for r in results], None),
]
for ax, (title, values, bound) in zip(axs, panels):
    ax.set_title(title)
    ax.set_xlabel('Time')
    ax.plot(t, values, 'b-')
    if bound is not None:
        ax.plot(t, bound, 'r--')
axs[3].set_ylim(bottom=0)
fig.subplots_adjust(hspace=0.7, top=0.93)
plt.show()

In [ ]:
# Heatmaps of curvatures: the optimal per-vertex target K* (gray where a
# vertex has no target), and the surface's Gaussian curvature inside and
# outside the tubes
fig, axs = plt.subplots(
    len(actual_curvatures), 3,
    figsize=(10, 3 * len(actual_curvatures)),
    gridspec_kw={'wspace': 0.},
)
axs = np.atleast_2d(axs)

# Compute color scale range that is wide enough to capture all target
# curvatures and is also centered at 0.
vmin = min(np.nanmin(target_curvatures), 0.)
vmax = max(np.nanmax(target_curvatures), 0.)
vmin = min(vmin, -vmax)
vmax = max(-vmin, vmax)

cmap = plt.cm.RdBu.copy()
cmap.set_bad(color='gray')
for row, (target_curvatures_i, actual_curvatures_i, axs_i) in enumerate(zip(target_curvatures, actual_curvatures, axs)):
    actual_curvatures_masked = np.copy(actual_curvatures_i)
    actual_curvatures_masked[np.isnan(target_curvatures_i)] = np.nan
    for ax, image in zip(axs_i, [target_curvatures_i, actual_curvatures_masked, actual_curvatures_i]):
        im = ax.imshow(image.T, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
        ax.set_xticks([])
        ax.set_yticks([])
    axs_i[0].set_ylabel(f'$t = {row}$')
    fig.colorbar(im, ax=axs_i[2])

axs[0][0].set_title('Optimal Target Curvatures')
axs[0][1].set_title('Actual Curvatures (tubes)')
axs[0][2].set_title('Actual Curvatures')

plt.show()